####Ex 1→ Create a DataFrame from scratch
The foundation of every pipeline. Create a DataFrame with an explicit schema using StructType — never rely on inferSchema in production as it's slow and can guess wrong.

In [0]:
from pyspark.sql.types import *

schema = StructType([
    StructField("emp_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("dept", StringType(), True),
    StructField("salary", DoubleType(), True),
    StructField("city", StringType(), True),
])

data = [
    (1, "Faizan", "IT", 60000, "Jaipur"),
    (2, "Ali", "Engineering", 70000, "Bikaner"),
    (3, "Sheikh", "Developer", 55000, "Delhi"),
]

df = spark.createDataFrame(data, schema)
df.show()
df.printSchema()

+------+------+-----------+-------+-------+
|emp_id|  name|       dept| salary|   city|
+------+------+-----------+-------+-------+
|     1|Faizan|         IT|60000.0| Jaipur|
|     2|   Ali|Engineering|70000.0|Bikaner|
|     3|Sheikh|  Developer|55000.0|  Delhi|
+------+------+-----------+-------+-------+

root
 |-- emp_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- dept: string (nullable = true)
 |-- salary: double (nullable = true)
 |-- city: string (nullable = true)



####Ex 2→ select() and column expressions
select() is how you pick and shape columns. You'll use it in every single pipeline. Know all three ways to reference a column.

In [0]:
from pyspark.sql.functions import *

# Referencing a particular columns
df.select(col("emp_name"),col("salary")).show()

# Rename a column with withColumnRenamed
df = (
    df.withColumnRenamed("name", "emp_name")
      .withColumnRenamed("dept", "department")
)
df.show()

# compute new column in the dataframe with the existing columns
df = (
    df.withColumn("department_upper", upper(col("department")))
      .withColumn("salary_after_hike", col("salary") * 1.10)
      .withColumn("tax_amount", col("salary") * 0.20)
      .withColumn("salary_after_tax", col("salary_after_hike") - col("tax_amount"))
)
df.show()

# deleting a column
df = df.drop("department_upper")
df.show()

####Ex 3→ filter() and where() with conditions
filter() and where() are identical in PySpark. Use them to subset rows. Know how to combine conditions and handle nulls.

In [0]:
from pyspark.sql.functions import *

# filter()
print("basic filter() applied:")
df.filter(col("department") == "Engineering").show()

# where()
print("where() applied:")
df.where(col("salary") > 60000).show()

# AND condition
print("AND condition applied:")
df.filter(
    (col("department") == "Engineering") & (col("salary") > 65000)
).show()

# OR condition
print("OR condition applied:")
df.filter(
    (col("city") == "Jaipur") | (col("city") == "Delhi")
).show()

# isin() → better than multiple OR conditions
print("isin() applied:")
df.filter(col("city").isin("Jaipur","Bikaner")).show()

# notin()
print("notin() applied:")
df.filter(~col("city").isin("Jaipur","Bikaner")).show()

# between() → Inclusive on both sides
print("between() applied:")
df.filter(col("salary").between(55000,70000)).show()

# Filter Nulls
print("Filter null values in department column")
df.filter(col("department").isNull()).show()
df.filter(col("department").isNotNull()).show()

####Ex 4→ withColumn() -add and modify columns
withColumn() adds a new column or replaces an existing one. It's how you engineer features and clean data in pipelines.

In [0]:
from pyspark.sql.functions import *

# Original table for reference
print("Original Table")
df.show()

# Add a new column
print("Add a new column...")
df.withColumn("salary_inr_lakh", round(col("salary")/ 100000, 2)).show()

# modify existing column (same name = replace)
print("Modify existing column...")
df.withColumn("name", upper(col("name"))).show()

# Conditional column with when/where
print("applying conditions on table...")
df.withColumn("salary_brand",
        when(col("salary") >= 100000, "High")
        .when(col("salary") >= 80000, "Mid")
        .otherwise("Low")
).show()

# cast column type
print("Type casting -changin column data type...")
df.withColumn("emp_id_str", col("emp_id").cast("string")).show()

Original Table
+------+------+-----------+-------+-------+
|emp_id|  name|       dept| salary|   city|
+------+------+-----------+-------+-------+
|     1|Faizan|         IT|60000.0| Jaipur|
|     2|   Ali|Engineering|70000.0|Bikaner|
|     3|Sheikh|  Developer|55000.0|  Delhi|
+------+------+-----------+-------+-------+

Add a new column...
+------+------+-----------+-------+-------+---------------+
|emp_id|  name|       dept| salary|   city|salary_inr_lakh|
+------+------+-----------+-------+-------+---------------+
|     1|Faizan|         IT|60000.0| Jaipur|            0.6|
|     2|   Ali|Engineering|70000.0|Bikaner|            0.7|
|     3|Sheikh|  Developer|55000.0|  Delhi|           0.55|
+------+------+-----------+-------+-------+---------------+

Modify existing column...
+------+------+-----------+-------+-------+
|emp_id|  name|       dept| salary|   city|
+------+------+-----------+-------+-------+
|     1|FAIZAN|         IT|60000.0| Jaipur|
|     2|   ALI|Engineering|70000.

####Ex 5→ withColumnRenamed() and drop()
Renaming and dropping columns is common when standardising raw data schemas or removing PII before writing.

In [0]:
from pyspark.sql.functions import *

# Original table for reference on changes
print("Original Table")
df.show()

# Rename single column
print("Renaming a single column from 'emp_id' to 'employee_id...'")
df.withColumnRenamed("emp_id", "employee_id").show()

# Rename multiple columns pairwise(chainwise)
print("Renaming multiple columns pairwise...")
df.withColumnRenamed("emp_id", "id") \
  .withColumnRenamed("dept", "department").show()

# Drop one column
print("Deleting single column 'city'")
df.drop("city").show()

# Drop multiple columns
print("Deleting multiple columns")
df.drop("city","emp_id").show()

Original Table
+------+------+-----------+-------+-------+
|emp_id|  name|       dept| salary|   city|
+------+------+-----------+-------+-------+
|     1|Faizan|         IT|60000.0| Jaipur|
|     2|   Ali|Engineering|70000.0|Bikaner|
|     3|Sheikh|  Developer|55000.0|  Delhi|
+------+------+-----------+-------+-------+

Renaming a single column from 'emp_id' to 'employee_id...'
+-----------+------+-----------+-------+-------+
|employee_id|  name|       dept| salary|   city|
+-----------+------+-----------+-------+-------+
|          1|Faizan|         IT|60000.0| Jaipur|
|          2|   Ali|Engineering|70000.0|Bikaner|
|          3|Sheikh|  Developer|55000.0|  Delhi|
+-----------+------+-----------+-------+-------+

Renaming multiple columns pairwise...
+---+------+-----------+-------+-------+
| id|  name| department| salary|   city|
+---+------+-----------+-------+-------+
|  1|Faizan|         IT|60000.0| Jaipur|
|  2|   Ali|Engineering|70000.0|Bikaner|
|  3|Sheikh|  Developer|55000.

####Ex 6→ Handling nulls - fillna, dropna, coalesce
Real data always has nulls. Know the three standard patterns: fill, drop, or coalesce with a fallback. This is mandatory in every ingestion pipeline.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Create a DataFrame with nulls for demo
# Demo table creation for nulls
null_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("dept", StringType(), True),
    StructField("salary", DoubleType(), True),
])

null_data = [
    (1, "Monish", "civil" , 95000.0),
    (2, "Tofik", "Marketing", None),
    (3, "Vikas", "Sales", 70000.0),
    (4, "Naseeb", "IT", None),
]
df_null = spark.createDataFrame(null_data, null_schema)
df_null.show()

# Drop rows where Any column is null
print("dropping rows where any column is null...")
df_null.dropna().show()

# Drop rows where ALL columns are null
print("dropping rows where all columns are null...")
df_null.dropna(how="all").show()

# Drop only if specific columns are null
print("specifying the columns to check for nulls...")
df_null.dropna(subset=["name","salary"]).show()

# control null positioning
print("doing null positioning as per our needs...")
df_null.sort(asc_nulls_last("salary")).show()
df_null.sort(asc_nulls_first("salary")).show()

+---+------+---------+-------+
| id|  name|     dept| salary|
+---+------+---------+-------+
|  1|Monish|    civil|95000.0|
|  2| Tofik|Marketing|   NULL|
|  3| Vikas|    Sales|70000.0|
|  4|Naseeb|       IT|   NULL|
+---+------+---------+-------+

dropping rows where any column is null...
+---+------+-----+-------+
| id|  name| dept| salary|
+---+------+-----+-------+
|  1|Monish|civil|95000.0|
|  3| Vikas|Sales|70000.0|
+---+------+-----+-------+

dropping rows where all columns are null...
+---+------+---------+-------+
| id|  name|     dept| salary|
+---+------+---------+-------+
|  1|Monish|    civil|95000.0|
|  2| Tofik|Marketing|   NULL|
|  3| Vikas|    Sales|70000.0|
|  4|Naseeb|       IT|   NULL|
+---+------+---------+-------+

specifying the columns to check for nulls...
+---+------+-----+-------+
| id|  name| dept| salary|
+---+------+-----+-------+
|  1|Monish|civil|95000.0|
|  3| Vikas|Sales|70000.0|
+---+------+-----+-------+

doing null positioning as per our needs...
+-

####Ex 7→ distinct(), dropDuplicates() and deduplication
Deduplication is one of the most common data quality tasks. Know the difference between distinct() and dropDuplicates().

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

# Creating DataFrame for duplicate values for using functions used
dup_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("dept", StringType(), True),
])

dup_data = [
    (1, "Alice", "Engineering"),
    (2, "Bob", "Marketing"),
    (1, "Alice", "Engineering"),  # full duplicate
    (3, "Alice", "HR"),           # same name, different record
    (2, "Bob", "Marketing"),      # full duplicate
]

df_dup = spark.createDataFrame(dup_data, dup_schema)
df_dup.show()

# distinct() - removes fully identical rows (all columns)
print("used distinct() function here...")
df_dup.distinct().show()

# dropDuplicates() - same as distinct() on all columns
print("used dropDuplicates() function here...")
df_dup.dropDuplicates().show()

# dropDuplicates() on specific records
print("used dropDuplicates() function on specific records...")
df_dup.dropDuplicates(["name"]).show()

# Check for duplicated before dedup
print("Checking for duplicates")
df_dup.groupBy("id", "name", "dept") \
      .agg(count("*").alias("count")) \
      .filter(col("count") > 1).show()

+---+-----+-----------+
| id| name|       dept|
+---+-----+-----------+
|  1|Alice|Engineering|
|  2|  Bob|  Marketing|
|  1|Alice|Engineering|
|  3|Alice|         HR|
|  2|  Bob|  Marketing|
+---+-----+-----------+

used distinct() function here...
+---+-----+-----------+
| id| name|       dept|
+---+-----+-----------+
|  1|Alice|Engineering|
|  2|  Bob|  Marketing|
|  3|Alice|         HR|
+---+-----+-----------+

used dropDuplicates() function here...
+---+-----+-----------+
| id| name|       dept|
+---+-----+-----------+
|  1|Alice|Engineering|
|  2|  Bob|  Marketing|
|  3|Alice|         HR|
+---+-----+-----------+

used dropDuplicates() function on specific records...
+---+-----+-----------+
| id| name|       dept|
+---+-----+-----------+
|  1|Alice|Engineering|
|  2|  Bob|  Marketing|
+---+-----+-----------+

Checking for duplicates
+---+-----+-----------+-----+
| id| name|       dept|count|
+---+-----+-----------+-----+
|  1|Alice|Engineering|    2|
|  2|  Bob|  Marketing|    2|


####Ex 8→ sort() and orderBy()
Sorting in PySpark. Know how to sort on multiple columns, control null positioning, and understand when sorting is expensive.

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

# Original table for reference on changes
print("Original Table...")
df.show()

# Basic sorting (defaulty ascending)
print("Sorting by salary in ascending order")
df.sort("salary").show()

# Basic Sorting (descending)
print("Sorting by salary in descending order")
df.sort(col("salary").desc()).show()

# Sort by multiple columns
print("Sorting by multiple columns...")
df.sort(
    col("dept").asc(),
    col("salary").desc()
).show()

# Sort then take Top N (common pattern)
print("sorting by taking top 2...")
df.orderBy(col("salary").desc()).limit(2).show()

Original Table...
+------+------+-----------+-------+-------+
|emp_id|  name|       dept| salary|   city|
+------+------+-----------+-------+-------+
|     1|Faizan|         IT|60000.0| Jaipur|
|     2|   Ali|Engineering|70000.0|Bikaner|
|     3|Sheikh|  Developer|55000.0|  Delhi|
+------+------+-----------+-------+-------+

Sorting by salary in ascending order
+------+------+-----------+-------+-------+
|emp_id|  name|       dept| salary|   city|
+------+------+-----------+-------+-------+
|     3|Sheikh|  Developer|55000.0|  Delhi|
|     1|Faizan|         IT|60000.0| Jaipur|
|     2|   Ali|Engineering|70000.0|Bikaner|
+------+------+-----------+-------+-------+

Sorting by salary in descending order
+------+------+-----------+-------+-------+
|emp_id|  name|       dept| salary|   city|
+------+------+-----------+-------+-------+
|     2|   Ali|Engineering|70000.0|Bikaner|
|     1|Faizan|         IT|60000.0| Jaipur|
|     3|Sheikh|  Developer|55000.0|  Delhi|
+------+------+----------